In [2]:
import random
import string
import numpy as np
from milvus import default_server

from pymilvus import (
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection, AnnSearchRequest, RRFRanker, connections,
)

from pymilvus.model.hybrid import BGEM3EmbeddingFunction
from sentence_transformers import SentenceTransformer

import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

# Add the EDP directory to the Python path
# sys.path.append(os.path.abspath(os.path.join('..', 'EDP')))

In [12]:
from pymilvus import connections, list_collections

# Connect to Milvus server (adjust the host and port accordingly)
connections.connect(alias="default", host="localhost", port="19530")

# List all existing collections
collections = list_collections()
print("Existing collections:", collections)


# Load the collection
collection = Collection("sbert_experiment")
# Get collection statistics
conn = collection._get_connection()
stats = conn.get_collection_stats(collection.name)
print(stats)

Existing collections: ['sbert_experiment', 'hybrid_experiment', 'hybrid_experiment2', 'sbert_experiment_test']
[key: "row_count"
value: "3620743"
]


In [3]:
from pymilvus import connections, list_collections

connections.connect("default", host="localhost", port="19530")

if connections.has_connection("default"):
    print("Successfully connected to Milvus")
else:
    print("Failed to connect to Milvus")


Successfully connected to Milvus


In [4]:
# Define the data schema for the new Collection
fields = [
    # Use provided id as primary key
    FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, max_length=1000),
    # Store the original text
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    # Store dense vectors
    FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=512),  # Ensure the dimension matches your embeddings
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'sbert_experiment_test'
col = Collection(col_name, schema, consistency_level="Strong")

In [5]:
dense_index = {"index_type": "FLAT", "metric_type": "IP"}
col.create_index("dense_vector", dense_index)
col.load()

In [6]:
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2', device='cuda')

/home/sbasir/Thesis/myenv3.10/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
def merge_and_chunk_text_fields(data, limit=128):
    chunked_data = []
    
    for item in data:
        # Merge the fields into 'text', separating them by " | "
        merged_text = " | ".join(filter(None, [item.get('text', ''), 
                                               item.get('provided_data', ''), 
                                               item.get('enriched_data', ''), 
                                               item.get('translated_data', '')]))
        
        # Apply the chunker function on the merged text
        chunks = chunker([merged_text], limit)
        
        # Append each chunk with an ID
        for i, chunk in enumerate(chunks):
            chunk_id = f'{item["id"]}_{i}'
            chunked_data.append({'id': chunk_id, 'text': chunk})
    
    return chunked_data


# Reusing your chunker function and adjusting it to take a limit as a parameter
def chunker(contexts: list, limit):
    chunks = []
    all_contexts = ' '.join(contexts).split('.')
    chunk = []
    for context in all_contexts:
        chunk.append(context)
        if len(chunk) >= 3 and len('.'.join(chunk)) > limit:
            # surpassed limit so add to chunks and reset
            chunks.append('.'.join(chunk).strip() + '.')
            # add some overlap between passages
            chunk = chunk[-2:]
    # if we finish and still have a chunk, add it
    if chunk:
        chunks.append('.'.join(chunk).strip() + '.')
    return chunks

def sbert_embeddings(batch_data):
    data = merge_and_chunk_text_fields(batch_data)
    print(data[0])
    embeddings = model.encode([x['text'] for x in data])

    # Prepare data for insertion
    entities = [
        [item['id'] for item in data], #IDs
        [item['text'] for item in data], #Text
        embeddings #Embeddings
    ]

    del embeddings

    return entities

In [8]:
import os
import gzip
import json
import time
import tqdm
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed

# Function to load data from a compressed JSON file
def load_compressed_json(file_path):
    """Function to load data from a compressed JSON file."""
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        return json.load(f)

# Function to load data in batches of 1000 documents for Milvus indexing
def load_data_in_batches_for_indexing(parsed_directory, batch_size=1000, max_workers=4):
    """Load data from compressed JSON files in batches for indexing into Milvus."""
    batch_data = []  # To store the current batch
    start_time = time.time()  # Record the start time

    # Collect all the .json.gz file paths
    file_paths = []
    for root, dirs, files in os.walk(parsed_directory):
        for file in files:
            if file.endswith('.json.gz'):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)

    total_files = len(file_paths)
    checkpoint_times = {}  # Dictionary to save checkpoint times

    # Use ThreadPoolExecutor to load files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {executor.submit(load_compressed_json, file_path): file_path for file_path in file_paths}

        for idx, future in enumerate(tqdm.tqdm(as_completed(future_to_file), total=total_files, desc="Loading and batching files")):
            file_path = future_to_file[future]
            try:
                data = future.result()  # Load the file's data
                
                # Add the data to the current batch
                batch_data.extend(data)  # Assuming data is a list of documents

                # If the current batch exceeds the batch size, process it
                if len(batch_data) >= batch_size:
                    # Call your Milvus insertion function here
                    index_batch_into_milvus(batch_data[:batch_size])  # Process 1000 documents at a time

                    # Remove the processed documents from the batch
                    batch_data = batch_data[batch_size:]

                # Record the time at each 10% completion
                percentage_complete = ((idx + 1) / total_files) * 100
                if percentage_complete >= 10 and (int(percentage_complete) % 10 == 0) and (int(percentage_complete) not in checkpoint_times):
                    elapsed_time = time.time() - start_time
                    checkpoint_times[int(percentage_complete)] = elapsed_time
                    print(f"Indexed {int(percentage_complete)}% of documents in {elapsed_time:.2f} seconds")

            except Exception as e:
                print(f"Error loading file {file_path}: {e}")

    # Process any remaining data in the last batch
    if batch_data:
        index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

    # Save checkpoint times to a file for future use
    with open('checkpoint_times.json', 'w') as f:
        json.dump(checkpoint_times, f)
    print("Checkpoint times saved to 'checkpoint_times.json'.")

def index_batch_into_milvus(batch_data):
    print(f"Indexing batch with {len(batch_data)} documents into Milvus...")

    entities = sbert_embeddings(batch_data)

    # Verify the lengths of each component to ensure they match
    print(f"Length of IDs: {len(entities[0])}")
    print(f"Length of texts: {len(entities[1])}")
    print(f"Shape of dense vectors: {len(entities[2])}")

    col.insert(entities)
    col.flush()

    del entities

    print("Batch indexed successfully.")

In [9]:
# run the function to load data in batches
load_data_in_batches_for_indexing('/home/sbasir/Thesis/Thesis/cp', max_workers=4)

Loading and batching files: 100%|██████████| 3/3 [00:00<00:00, 9946.97it/s]


Indexed 100% of documents in 0.02 seconds
Indexing batch with 835 documents into Milvus...
{'id': '/1/10107_3101173_0', 'text': 'timestamp_update is 2019-09-06T10:52:04.496Z | type is TEXT | content_tier is 2 | metadata_tier is 0 | edm:dataProvider is National Library of Wales | edm:provider is National Library of Wales | dc:language is eng | dc:title is 1865-10-11 - Potter&apos;s electric news | dc:type is Text | Newspaper Issue | dc:language is eng.'}
Length of IDs: 1316
Length of texts: 1316
Shape of dense vectors: 1316
Batch indexed successfully.
Checkpoint times saved to 'checkpoint_times.json'.


In [13]:
query = "Vermeer"
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
query_embeddings = model.encode(query)
k=10

# display(query_embeddings)

# print(query_embeddings)

search_params = {"metric_type": "IP"}

# Search topK docs based on dense and sparse vectors and rerank with RRF.
res = col.search(
    data = [query_embeddings],
    anns_field="dense_vector",
    param=search_params,
    limit=10,
    output_fields=["pk","text"]
)

for result in res[0]:
    print(result)

/home/sbasir/Thesis/myenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


id: /10/id_gvn_KONB14Borms0337_5, distance: 0.12719784677028656, entity: {'pk': '/10/id_gvn_KONB14Borms0337_5', 'text': 'Hij sleurt de slang naar zijn kamp, en ontdekt dan dat de slang een krokodil is geworden: de poten van de hond steken door diens vel. Bevat tweeregelige onderschriften in Nederlands en Frans. | dc:subject is Eene gedaanteverandering | Ontdekkingsreizigers | Literatuur | Tweetalige uitgaven | Kunst en cultuur | Centsprenten | dc:title is Eene gedaanteverandering | dcterms:created is tussen 1911-1935 | dcterms:medium is 1 litho | edm:currentLocation is Turnhout.'}
id: /10/id_gvn_KONB14SMC_K0059_1, distance: 0.11103351414203644, entity: {'pk': '/10/id_gvn_KONB14SMC_K0059_1', 'text': "851Z | type is IMAGE | content_tier is 4 | metadata_tier is A | edm:dataProvider is KB, National Library of the Netherlands | Koninklijke Bibliotheek | edm:provider is KB, National Library of the Netherlands | Koninklijke Bibliotheek | dc:description is Centsprent met 4 x 4 niet omkaderde h